In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
import state_store as ss
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Apply lakebase_schema.sql (control, checkpoints, reseed queue, telemetry) in ONE transaction:
# the pending renames (state_store.RENAMES: ct_checkpoint -> checkpoint, sweep_run -> ingest_run, ...),
# every DDL statement, then the schema version (state_store.SCHEMA_VERSION). Re-applying is a no-op,
# and a failure leaves the database unchanged. Ingestion runs apply a pending migration the same way
# (StateStore.migrate), so running this after an upgrade is optional.
schema_path = "/Workspace" + nb_dir + "/lakebase_schema.sql"
with open(schema_path) as f:
    ddl = f.read()
before = store.schema_version()
result = store.migrate(ddl, force=True)
for stmt in result["renames"]:
    print(stmt)
print(f"applied {len(result['renames'])} renames and {len(ss.split_sql_statements(ddl))} DDL statements "
      f"to {db_name}: schema version {before} -> {ss.SCHEMA_VERSION}")